In [2]:
import pandas as pd
import numpy as np
from extract import extract_all

In [4]:
data=extract_all()

can not load extract.py table due to an errorunsupported operand type(s) for +: 'NoneType' and 'str'
can not load load.py table due to an errorunsupported operand type(s) for +: 'NoneType' and 'str'
can not load note_core.ipynb table due to an errorunsupported operand type(s) for +: 'NoneType' and 'str'
can not load note_mart.ipynb table due to an errorunsupported operand type(s) for +: 'NoneType' and 'str'
can not load transform_core.py table due to an errorunsupported operand type(s) for +: 'NoneType' and 'str'
can not load transform_mart.py table due to an errorunsupported operand type(s) for +: 'NoneType' and 'str'
can not load transform_staging.py table due to an errorunsupported operand type(s) for +: 'NoneType' and 'str'
can not load utils.py table due to an errorunsupported operand type(s) for +: 'NoneType' and 'str'
can not load __init__.py table due to an errorunsupported operand type(s) for +: 'NoneType' and 'str'
can not load __pycache__ table due to an errorunsupported ope

In [4]:
print(data.keys())

dict_keys(['customers', 'geolocation', 'orders', 'order_items', 'order_payments', 'order_reviews', 'products', 'sellers', 'product_category_name_translation'])


In [5]:
for name,df in data.items():
    print(name)
    for colomn in df.columns:
        print(f"{colomn} : {df[colomn].dtype}")

customers
customer_id : object
customer_unique_id : object
customer_zip_code_prefix : int64
customer_city : object
customer_state : object
geolocation
geolocation_zip_code_prefix : int64
geolocation_lat : float64
geolocation_lng : float64
geolocation_city : object
geolocation_state : object
orders
order_id : object
customer_id : object
order_status : object
order_purchase_timestamp : object
order_approved_at : object
order_delivered_carrier_date : object
order_delivered_customer_date : object
order_estimated_delivery_date : object
order_items
order_id : object
order_item_id : int64
product_id : object
seller_id : object
shipping_limit_date : object
price : float64
freight_value : float64
order_payments
order_id : object
payment_sequential : int64
payment_type : object
payment_installments : int64
payment_value : float64
order_reviews
review_id : object
order_id : object
review_score : int64
review_comment_title : object
review_comment_message : object
review_creation_date : object
revi

In [6]:
data['orders'].head(3)
data['orders']['order_purchase_timestamp']=pd.to_datetime(data['orders']['order_purchase_timestamp'])
data['orders']['order_approved_at']=pd.to_datetime(data['orders']['order_approved_at'])
data['orders']['order_delivered_carrier_date']=pd.to_datetime(data['orders']['order_delivered_carrier_date'])
data['orders']['order_delivered_customer_date']=pd.to_datetime(data['orders']['order_delivered_customer_date'])
data['orders']['order_estimated_delivery_date']=pd.to_datetime(data['orders']['order_estimated_delivery_date'])
data['orders']['delivery_days']=(data['orders']['order_delivered_customer_date']-data['orders']['order_purchase_timestamp']).dt.days
data['orders']=data['orders'].dropna(subset='order_id')
data['orders']=data['orders'][data['orders']['order_status']!='unavailable']

In [7]:
data['orders']['delivery_days']=(data['orders']['order_delivered_customer_date']-data['orders']['order_purchase_timestamp']).dt.days
data['orders']=data['orders'].dropna(subset='order_id')
data['orders']=data['orders'][data['orders']['order_status']!='unavailable']


In [8]:

data['orders'].groupby('order_status').groups.keys()

dict_keys(['approved', 'canceled', 'created', 'delivered', 'invoiced', 'processing', 'shipped'])

In [9]:
data['order_items']=data['order_items'][data['order_items']['freight_value']>0]
data['order_items']=data['order_items'][data['order_items']['price']>0]
data['order_items']=data['order_items'].dropna(subset=['order_id','product_id'])

In [10]:
data['order_payments'].head(5)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [11]:
data['order_payments']=data['order_payments'].groupby('order_id').agg(
    total_price=('payment_value', 'sum'),
    common_type=('payment_type', pd.Series.mode)
).reset_index()

In [12]:
data['order_reviews']=data['order_reviews'].sort_values('order_id')
data['order_reviews']=data['order_reviews'].drop_duplicates(subset='order_id',keep='last')
data['order_reviews']['review_score']=data['order_reviews']['review_score'].fillna(data['order_reviews']['review_score'].median)
data['order_reviews']=data['order_reviews'].dropna(subset='review_comment_title')

In [13]:
data['customers']=data['customers'].drop_duplicates(subset='customer_unique_id')
data['customers']['customer_city']=data['customers']['customer_city'].str.title()
data['customers']['customer_state']=data['customers']['customer_state'].str.upper()


In [14]:
data['products']=pd.merge(data['products'],data['product_category_name_translation'],on='product_category_name')
data['products']['product_category_name']=data['products']['product_category_name'].fillna('Unknown')
col=['product_name_lenght','product_description_lenght','product_photos_qty','product_weight_g','product_length_cm','product_height_cm','product_width_cm']
data['products'][col]=data['products'][col].fillna(data['products'][col].median)

In [15]:
data['sellers']['seller_city']=data['sellers']['seller_city'].str.title()
data['sellers']['seller_state']=data['sellers']['seller_state'].str.upper()
data['sellers']=data['sellers'].drop_duplicates(subset='seller_id')
data['sellers'].head(5)

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,Campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,Mogi Guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,Rio De Janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,Sao Paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,Braganca Paulista,SP


In [16]:
data['dim_date'] = pd.DataFrame({
    'date': data['orders']['order_purchase_timestamp']
})
data['dim_date']['date']=pd.to_datetime(data['dim_date']['date'])
data['dim_date']['year']=data['dim_date']['date'].dt.year
data['dim_date']['quarter']=data['dim_date']['date'].dt.quarter
data['dim_date']['month']=data['dim_date']['date'].dt.month
data['dim_date']['month_name']=data['dim_date']['date'].dt.month_name()
data['dim_date']['week']=data['dim_date']['date'].dt.weekday
data['dim_date']['week_day']=data['dim_date']['date'].dt.day_of_week
data['dim_date']['day']=data['dim_date']['date'].dt.day_name()
data['dim_date']['is_weekend']=data['dim_date']['day'].isin(['Saturday','Sunday'])
data['dim_date'].head(5)

,date,year,quarter,month,month_name,week,week_day,day,is_weekend
0,2017-10-02 10:56:33,2017,4,10,October,0,0,Monday,False
1,2018-07-24 20:41:37,2018,3,7,July,1,1,Tuesday,False
2,2018-08-08 08:38:49,2018,3,8,August,2,2,Wednesday,False
3,2017-11-18 19:28:06,2017,4,11,November,5,5,Saturday,True
4,2018-02-13 21:18:39,2018,1,2,February,1,1,Tuesday,False


In [17]:
data['products'].iloc[:410]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,housewares
...,...,...,...,...,...,...,...,...,...,...
405,c5c66926b3f964b8cf9e7dfb1150d13e,esporte_lazer,41.0,128.0,1.0,200.0,21.0,9.0,13.0,sports_leisure
406,3488d2ce36e718097c1509444289ef7f,perfumaria,40.0,1176.0,1.0,514.0,18.0,12.0,15.0,perfumery
407,a0ff95d002b9e836b60b47e97a200f93,casa_construcao,37.0,426.0,2.0,3100.0,38.0,28.0,28.0,home_construction
408,068ceaa8a3d9d385cbf53d5335f89f80,utilidades_domesticas,42.0,395.0,1.0,1200.0,27.0,18.0,23.0,housewares


In [18]:
products = data['products']

print(products.iloc[408])

product_id                       068ceaa8a3d9d385cbf53d5335f89f80
product_category_name                       utilidades_domesticas
product_name_lenght                                          42.0
product_description_lenght                                  395.0
product_photos_qty                                            1.0
product_weight_g                                           1200.0
product_length_cm                                            27.0
product_height_cm                                            18.0
product_width_cm                                             23.0
product_category_name_english                          housewares
Name: 408, dtype: object
